In [1]:
def load_yearly_signals(year, buys_path_template='buys_{}.csv', sells_path_template='sells_{}.csv'):
    """
    Load buy and sell signals for a specific year.
    
    Parameters:
    -----------
    year : int
        Year to load signals for
    buys_path_template : str
        Template for buys file path (use {} for year placeholder)
    sells_path_template : str
        Template for sells file path (use {} for year placeholder)
    
    Returns:
    --------
    permno_set : set
        Set of permnos in the buy and sell signals for this year
    """
    try:
        buys = pd.read_csv(buys_path_template.format(year), index_col=1)
        sells = pd.read_csv(sells_path_template.format(year), index_col=1)
        
        buys.index.name = 'permno'
        sells.index.name = 'permno'
        
        buys_index = buys.index.astype(int)
        sells_index = sells.index.astype(int)
        
        return set(buys_index.union(sells_index))
    except FileNotFoundError as e:
        print(f"  ⚠ Warning: Could not load signals for year {year}: {e}")
        return set()

def load_finbert_signals(signals_path):
    """
    Load FinBERT monthly signals from CSV file.
    
    Parameters:
    -----------
    signals_path : str
        Path to monthly_signals.csv file
    
    Returns:
    --------
    signals_df : pd.DataFrame
        DataFrame with columns: symbol, company, year_month, signal, avg_sentiment_score
    """
    try:
        signals_df = pd.read_csv(signals_path)
        # Convert year_month to datetime (end of month)
        signals_df['date'] = pd.to_datetime(signals_df['year_month']) + pd.offsets.MonthEnd(0)
        return signals_df
    except FileNotFoundError as e:
        print(f"  ⚠ Warning: Could not load FinBERT signals: {e}")
        return pd.DataFrame(columns=['symbol', 'company', 'year_month', 'signal', 'date'])

def get_finbert_permnos_for_date(signals_df, ticker_to_permno, date):
    """
    Get set of permnos with 'buy' or 'sell' signals for a specific date.
    
    Parameters:
    -----------
    signals_df : pd.DataFrame
        FinBERT signals dataframe
    ticker_to_permno : dict
        Mapping from ticker symbol to permno
    date : pd.Timestamp
        Date to get signals for
    
    Returns:
    --------
    permno_set : set
        Set of permnos with buy or sell signals on this date
    """
    # Get signals for this date
    date_signals = signals_df[signals_df['date'] == date]
    
    # Filter for buy and sell signals (exclude hold)
    buy_signals = date_signals[date_signals['signal'] == 'buy']
    sell_signals = date_signals[date_signals['signal'] == 'sell']
    
    # Convert tickers to permnos
    permnos = set()
    for ticker in buy_signals['symbol'].values:
        if ticker in ticker_to_permno:
            permnos.add(ticker_to_permno[ticker])
    for ticker in sell_signals['symbol'].values:
        if ticker in ticker_to_permno:
            permnos.add(ticker_to_permno[ticker])
    
    return permnos


def create_ticker_to_permno_mapping(df):
    """
    Create a mapping from ticker to permno from the returns dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Returns dataframe with 'ticker' and 'permno' columns
    
    Returns:
    --------
    ticker_to_permno : dict
        Mapping from ticker to permno (uses most recent permno for each ticker)
    """
    if 'ticker' not in df.columns:
        raise ValueError("DataFrame must have 'ticker' column for mapping")
    
    # Drop NaN tickers
    valid_df = df[df['ticker'].notna()].copy()
    
    # Get the most recent permno for each ticker
    ticker_to_permno = valid_df.groupby('ticker')['permno'].last().to_dict()
    
    return ticker_to_permno

def load_analyst_recommendations(rec_changes_path):
    """
    Load significant recommendation changes from CSV file.
    
    Parameters:
    -----------
    rec_changes_path : str
        Path to significant_recommendation_changes.csv file
    
    Returns:
    --------
    rec_changes_df : pd.DataFrame
        DataFrame with columns: permno, date, ticker, mean_recommendation, 
        recommendation_change, num_recommendations
    """
    try:
        rec_changes_df = pd.read_csv(rec_changes_path)
        rec_changes_df['date'] = pd.to_datetime(rec_changes_df['date'])
        rec_changes_df['permno'] = rec_changes_df['permno'].astype(int)
        return rec_changes_df
    except FileNotFoundError as e:
        print(f"  ⚠ Warning: Could not load recommendation changes: {e}")
        return pd.DataFrame(columns=['permno', 'date', 'ticker', 'mean_recommendation', 
                                    'recommendation_change', 'num_recommendations'])


def get_signal_permnos_for_date(rec_changes_df, date, buy_threshold=-0.5, sell_threshold=0.5):
    """
    Get sets of permnos with buy/sell signals based on recommendation changes.
    
    Note: Negative change = upgrade (moving toward Strong Buy) = BUY signal
          Positive change = downgrade (moving toward Sell) = SELL signal
    
    Parameters:
    -----------
    rec_changes_df : pd.DataFrame
        Recommendation changes dataframe
    date : pd.Timestamp
        Date to get signals for
    buy_threshold : float
        Threshold for buy signals (default: -0.5)
    sell_threshold : float
        Threshold for sell signals (default: 0.5)
    
    Returns:
    --------
    buy_permnos : set
        Set of permnos with buy signals
    sell_permnos : set
        Set of permnos with sell signals
    """
    date_changes = rec_changes_df[rec_changes_df['date'] == date]
    
    # Buy signals: negative changes (recommendations getting better)
    buys = date_changes[date_changes['recommendation_change'] <= buy_threshold]
    buy_permnos = set(buys['permno'].values)
    
    # Sell signals: positive changes (recommendations getting worse)
    sells = date_changes[date_changes['recommendation_change'] >= sell_threshold]
    sell_permnos = set(sells['permno'].values)
    
    return buy_permnos | sell_permnos

In [5]:
import pandas as pd

def analyze_signal_disagreements_with_poet_filter(
    df, 
    test_start_date='2021-10-31', 
    test_end_date='2024-04-30',
    lookback_window=180,
    buys_path_template='buys_{}.csv',
    sells_path_template='sells_{}.csv',
    analyst_rec_path='../examples/monthly_mean_recommendations_decay.csv',
    finbert_signals_path='../examples/monthly_signals_decay.csv'
):
    """
    Analyzes pairwise signal agreements and disagreements on the exact 
    stock universe available to POET (15-year continuous return history).
    Applies the POET p < 2 liquidation rule (treats 1-stock portfolios as 0).
    """
    print("Creating ticker to permno mapping...")
    ticker_to_permno = create_ticker_to_permno_mapping(df)
    
    print("Loading datasets...")
    finbert_df = load_finbert_signals(finbert_signals_path)
    analyst_df = load_analyst_recommendations(analyst_rec_path)
    
    df['datadate'] = pd.to_datetime(df['datadate'])
    all_dates = sorted(df['datadate'].unique())
    
    test_start_dt = pd.to_datetime(test_start_date)
    test_end_dt = pd.to_datetime(test_end_date)
    
    try:
        test_start_idx = all_dates.index(test_start_dt)
        test_end_idx = all_dates.index(test_end_dt)
    except ValueError as e:
        raise ValueError(f"Date not found in DataFrame: {e}")
        
    results = []
    yearly_signals_cache = {}
    
    print("Analyzing signal overlaps with 15-year continuous data requirement...\n")
    
    for t in range(test_start_idx, test_end_idx + 1):
        current_date = all_dates[t]
        current_year = current_date.year
        
        # 1. POET Continuous Data Filter (15 years / 180 months)
        if t < lookback_window:
            print(f"Skipping {current_date.date()}: Not enough historical data.")
            continue
            
        window_start_date = all_dates[t - lookback_window]
        window_end_date = all_dates[t - 1]
        
        # Find stocks with zero missing returns in the lookback window
        window_data = df[(df['datadate'] >= window_start_date) & 
                         (df['datadate'] <= window_end_date)]
        obs_counts = window_data.groupby('permno')['ret_fwd_1'].count()
        poet_valid_permnos = set(obs_counts[obs_counts == lookback_window].index)
        
        # 2. Load Signals (Intersected with POET universe)
        if current_year not in yearly_signals_cache:
            yearly_signals_cache[current_year] = load_yearly_signals(
                current_year, buys_path_template, sells_path_template
            )
        llm_permnos = yearly_signals_cache[current_year] & poet_valid_permnos
        
        human_permnos = set()
        if analyst_df is not None and len(analyst_df) > 0:
            human_permnos = get_signal_permnos_for_date(analyst_df, current_date) & poet_valid_permnos

        finbert_permnos = set()
        if finbert_df is not None and len(finbert_df) > 0:
            finbert_permnos = get_finbert_permnos_for_date(finbert_df, ticker_to_permno, current_date) & poet_valid_permnos
            
        # 3. Disjoint Partitioning of the 2-out-of-3 Universe
        all_three = human_permnos & llm_permnos & finbert_permnos
        human_llm_no_finbert = (human_permnos & llm_permnos) - finbert_permnos
        finbert_llm_no_human = (finbert_permnos & llm_permnos) - human_permnos
        finbert_human_no_llm = (finbert_permnos & human_permnos) - llm_permnos
        
        # Portfolio pool is the union of all 4 subsets
        portfolio_pool = all_three | human_llm_no_finbert | finbert_llm_no_human | finbert_human_no_llm
        
        # 4. Enforce POET's p < 2 rule: Treat 1-stock portfolio as 0
        if len(portfolio_pool) <= 1:
            portfolio_pool = set()
            all_three = set()
            human_llm_no_finbert = set()
            finbert_llm_no_human = set()
            finbert_human_no_llm = set()
            
        p_size = len(portfolio_pool)
        
        results.append({
            'date': current_date,
            'poet_eligible_universe_size': len(poet_valid_permnos),
            'final_portfolio_size': p_size,
            
            # --- 1. Human + LLM (No FinBERT) ---
            'human_llm_no_fb_count': len(human_llm_no_finbert),
            'human_llm_no_fb_pct': (len(human_llm_no_finbert) / p_size) if p_size > 0 else 0.0,
            'human_llm_no_fb_permnos': list(human_llm_no_finbert),
            
            # --- 2. FinBERT + LLM (No Humans) ---
            'fb_llm_no_human_count': len(finbert_llm_no_human),
            'fb_llm_no_human_pct': (len(finbert_llm_no_human) / p_size) if p_size > 0 else 0.0,
            'fb_llm_no_human_permnos': list(finbert_llm_no_human),
            
            # --- 3. FinBERT + Human (No LLM) ---
            'fb_human_no_llm_count': len(finbert_human_no_llm),
            'fb_human_no_llm_pct': (len(finbert_human_no_llm) / p_size) if p_size > 0 else 0.0,
            'fb_human_no_llm_permnos': list(finbert_human_no_llm),
            
            # --- 4. All Three Agree ---
            'all_three_agree_count': len(all_three),
            'all_three_agree_pct': (len(all_three) / p_size) if p_size > 0 else 0.0,
            'all_three_agree_permnos': list(all_three)
        })
        
    results_df = pd.DataFrame(results)
    
    # Summary Statistics
    active_months = results_df[results_df['final_portfolio_size'] > 0]
    
    print("="*75)
    print("HUMAN-LLM-FINBERT PORTFOLIO COMPREHENSIVE SIGNAL BREAKDOWN")
    print("="*75)
    print(f"Total Months Evaluated: {len(results_df)} | Months with Active Portfolio (p >= 2): {len(active_months)}")
    print(f"Average Active Portfolio Size: {active_months['final_portfolio_size'].mean():.1f} stocks\n")
    
    print(f"{'Signal Combination':<32} | {'Avg Stocks/Mo':<14} | {'Avg % of Portfolio':<18}")
    print("-" * 75)
    print(f"{'Human + LLM (No FinBERT)':<32} | {active_months['human_llm_no_fb_count'].mean():>10.2f}    | {active_months['human_llm_no_fb_pct'].mean()*100:>14.1f}%")
    print(f"{'FinBERT + LLM (No Human)':<32} | {active_months['fb_llm_no_human_count'].mean():>10.2f}    | {active_months['fb_llm_no_human_pct'].mean()*100:>14.1f}%")
    print(f"{'FinBERT + Human (No LLM)':<32} | {active_months['fb_human_no_llm_count'].mean():>10.2f}    | {active_months['fb_human_no_llm_pct'].mean()*100:>14.1f}%")
    print(f"{'All 3 Agree (Consensus)':<32} | {active_months['all_three_agree_count'].mean():>10.2f}    | {active_months['all_three_agree_pct'].mean()*100:>14.1f}%")
    print("="*75)
    
    return results_df

In [3]:
df = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
df['ret_fwd_1'] = df.groupby('permno')['ret_excess'].shift(-1)

In [6]:
# Example execution to match your backtest exact setup:
disagreement_df = analyze_signal_disagreements_with_poet_filter(
    df,
    test_start_date='2021-10-31',
    test_end_date='2024-04-30',
    lookback_window=180,
    buys_path_template='../GPT 3.5/Run 1 (Main)/buys_{}.csv',
    sells_path_template='../GPT 3.5/Run 1 (Main)/sells_{}.csv',
    analyst_rec_path='../examples/monthly_mean_recommendations_decay.csv',
    finbert_signals_path='../examples/monthly_signals_decay.csv'
)

Creating ticker to permno mapping...
Loading datasets...
Analyzing signal overlaps with 15-year continuous data requirement...

HUMAN-LLM-FINBERT PORTFOLIO COMPREHENSIVE SIGNAL BREAKDOWN
Total Months Evaluated: 31 | Months with Active Portfolio (p >= 2): 28
Average Active Portfolio Size: 7.3 stocks

Signal Combination               | Avg Stocks/Mo  | Avg % of Portfolio
---------------------------------------------------------------------------
Human + LLM (No FinBERT)         |       3.61    |           49.1%
FinBERT + LLM (No Human)         |       0.57    |            6.3%
FinBERT + Human (No LLM)         |       2.75    |           40.1%
All 3 Agree (Consensus)          |       0.39    |            4.6%


In [18]:
disagreement_df

,date,poet_eligible_universe_size,final_portfolio_size,human_llm_no_fb_count,human_llm_no_fb_pct,human_llm_no_fb_permnos,fb_llm_no_human_count,fb_llm_no_human_pct,fb_llm_no_human_permnos,fb_human_no_llm_count,fb_human_no_llm_pct,fb_human_no_llm_permnos,all_three_agree_count,all_three_agree_pct,all_three_agree_permnos
0,2021-10-31,253,7,5,0.714286,"[59176, 11850, 66093, 14541, 59408]",0,0.000000,[],2,0.285714,"[34833, 86356]",0,0.000000,[]
1,2021-11-30,257,6,5,0.833333,"[11850, 14541, 38703, 59408, 12060]",0,0.000000,[],1,0.166667,[77661],0,0.000000,[]
2,2021-12-31,255,5,4,0.800000,"[59176, 66093, 14702, 38703]",0,0.000000,[],1,0.200000,[53613],0,0.000000,[]
3,2022-01-31,256,6,2,0.333333,"[86868, 15069]",0,0.000000,[],4,0.666667,"[57568, 89195, 89525, 40272]",0,0.000000,[]
4,2022-02-28,255,4,3,0.750000,"[25081, 86868, 15069]",0,0.000000,[],1,0.250000,[35044],0,0.000000,[]
5,2022-03-31,257,5,3,0.600000,"[38703, 15069, 81055]",0,0.000000,[],2,0.400000,"[40272, 27828]",0,0.000000,[]
6,2022-04-30,259,8,4,0.500000,"[25081, 34746, 15069, 81055]",0,0.000000,[],4,0.500000,"[65875, 62092, 16678, 27991]",0,0.000000,[]
7,2022-05-31,259,0,0,0.000000,[],0,0.000000,[],0,0.000000,[],0,0.000000,[]
8,2022-06-30,259,6,4,0.666667,"[59408, 86868, 15069, 38703]",0,0.000000,[],2,0.333333,"[39642, 22509]",0,0.000000,[]
9,2022-07-31,260,7,3,0.428571,"[59408, 86868, 15069]",0,0.000000,[],4,0.571429,"[22592, 49680, 65875, 18542]",0,0.000000,[]
